# SQL reference 11: DDL, constraints, views and modeling

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/sql/sql-reference-11-ddl-constraints-and-views.ipynb)

`CREATE TABLE` options (defaults, partitioning, clustering, managed and external tables), inspecting tables, `ALTER TABLE` and schema evolution, enforced and informational constraints, data-quality checks in SQL, views and caching, and grain, facts, dimensions and surrogate keys. Practise it in [SQL practice 11: DDL, constraints, views and modeling](sql-practice-11-ddl-constraints-and-views.ipynb).

*Last reviewed: November 2026, against Apache Spark 4.0 (Spark SQL).*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q pyspark~=4.0.0 delta-spark~=4.0.0

## Setup and sample data

A local Spark session, the shared tables as temporary views: `customers`, `orders`, `order_items`, `products`, `events` and `employees` (customer 4 has no region, customers 7 and 8 never ordered, `SKU-4014` has no product row, one event is an exact retry, two pairs of employees tie on salary, Dana out-earns her manager and Jo has no department), and three helpers: `show(query)` runs a query and prints its rows untruncated, `plan(query)` returns what `EXPLAIN` prints, and `expected_error()` prints the error class of an example that fails on purpose. This notebook uses a Delta Lake session (the Delta jars download on first use) so constraints are enforced. `fresh(table, select)` re-creates a Delta table and `drop(*tables)` removes tables.

In [ ]:
import os
import sys
import tempfile
import time

from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable  # Python workers use this notebook's interpreter
os.environ["TZ"] = "UTC"  # the sample timestamps are UTC
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="sql-")  # tables created in this notebook live here

builder = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")  # Delta's SQL: MERGE, UPDATE, time travel
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.shuffle.partitions", "4")  # the default of 200 is far too many for tiny data
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()  # fetches the Delta jars on the first run
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

In [ ]:
# Shared sample tables as temporary views. Each statement is ordinary Spark SQL.
TABLES = {
    "customers": """CREATE OR REPLACE TEMP VIEW customers AS SELECT * FROM VALUES
  (1, 'Acme Corp',      'North', DATE'2024-01-03'),
  (2, 'Bright Retail',  'North', DATE'2024-02-11'),
  (3, 'Cyan Traders',   'South', DATE'2024-03-15'),
  (4, 'Delta Grocers',  NULL,    DATE'2024-05-02'),   -- region never arrived from the CRM
  (5, 'Echo Supplies',  'West',  DATE'2024-06-20'),
  (6, 'Foxtrot Foods',  'West',  DATE'2024-08-01'),
  (7, 'Gamma Sports',   'East',  DATE'2024-09-14'),   -- 7 and 8 have never ordered
  (8, 'Hotel Homeware', 'East',  DATE'2024-11-05')
AS customers(customer_id, name, region, signup_date)""",
    "orders": """CREATE OR REPLACE TEMP VIEW orders AS SELECT order_id, customer_id, status, order_ts, CAST(amount AS DECIMAL(12,2)) AS amount FROM VALUES
  (1,  1, 'paid',     TIMESTAMP'2026-01-05 09:15:00', 120.00),
  (2,  1, 'shipped',  TIMESTAMP'2026-01-05 10:05:00', 84.50),
  (3,  1, 'returned', TIMESTAMP'2026-01-05 11:40:00', 30.00),
  (4,  2, 'paid',     TIMESTAMP'2026-01-06 15:20:00', 260.00),
  (5,  2, 'placed',   TIMESTAMP'2026-01-07 08:05:00', 55.25),
  (6,  3, 'shipped',  TIMESTAMP'2026-01-07 17:45:00', 149.00),
  (7,  3, 'paid',     TIMESTAMP'2026-01-09 12:10:00', 310.00),
  (8,  4, 'returned', TIMESTAMP'2026-01-10 19:30:00', 89.90),
  (9,  4, 'shipped',  TIMESTAMP'2026-01-12 08:55:00', 1180.00),
  (10, 5, 'paid',     TIMESTAMP'2026-01-12 13:00:00', 74.00),
  (11, 6, 'placed',   TIMESTAMP'2026-01-13 10:25:00', 210.00),
  (12, 6, 'shipped',  TIMESTAMP'2026-01-14 00:30:00', 96.40)   -- just after midnight UTC
AS orders(order_id, customer_id, status, order_ts, amount)""",
    "order_items": """CREATE OR REPLACE TEMP VIEW order_items AS SELECT order_id, sku, qty, CAST(unit_price AS DECIMAL(8,2)) AS unit_price FROM VALUES
  (1, 'SKU-4001', 2, 12.50), (1, 'SKU-4007', 1, 95.00), (2, 'SKU-4002', 1, 34.50), (2, 'SKU-4009', 2, 25.00),
  (3, 'SKU-4002', 1, 30.00), (4, 'SKU-4003', 1, 149.00), (4, 'SKU-4004', 2, 55.50), (5, 'SKU-4005', 1, 55.25),
  (6, 'SKU-4003', 1, 149.00), (7, 'SKU-4006', 1, 189.00), (7, 'SKU-4008', 1, 121.00), (8, 'SKU-4010', 1, 89.90),
  (9, 'SKU-4011', 20, 59.00), (10, 'SKU-4012', 2, 37.00), (11, 'SKU-4013', 1, 210.00),
  (12, 'SKU-4014', 1, 41.40), (12, 'SKU-4005', 1, 55.00)
AS order_items(order_id, sku, qty, unit_price)""",
    "products": """CREATE OR REPLACE TEMP VIEW products AS SELECT sku, category, CAST(list_price AS DECIMAL(8,2)) AS list_price FROM VALUES
  ('SKU-4001', 'Kitchen', 12.50), ('SKU-4002', 'Kitchen', 34.50), ('SKU-4003', 'Garden', 149.00),
  ('SKU-4004', 'Garden', 55.50), ('SKU-4005', 'Pantry', 55.25), ('SKU-4006', 'Outdoor', 189.00),
  ('SKU-4007', 'Kitchen', 95.00), ('SKU-4008', 'Outdoor', 121.00), ('SKU-4009', 'Pantry', 25.00),
  ('SKU-4010', 'Garden', 89.90), ('SKU-4011', 'Pantry', 59.00), ('SKU-4012', 'Kitchen', 37.00),
  ('SKU-4013', 'Outdoor', 210.00)   -- SKU-4014 has no product row
AS products(sku, category, list_price)""",
    "events": """CREATE OR REPLACE TEMP VIEW events AS SELECT * FROM VALUES
  (1, 'view', TIMESTAMP'2026-01-06 09:00:00'), (1, 'add_to_cart', TIMESTAMP'2026-01-06 09:02:00'),
  (1, 'purchase', TIMESTAMP'2026-01-06 09:10:00'), (2, 'view', TIMESTAMP'2026-01-07 10:00:00'),
  (2, 'purchase', TIMESTAMP'2026-01-07 10:40:00'), (3, 'view', TIMESTAMP'2026-01-06 11:15:00'),
  (3, 'view', TIMESTAMP'2026-01-08 11:20:00'), (4, 'add_to_cart', TIMESTAMP'2026-01-09 20:05:00'),
  (4, 'purchase', TIMESTAMP'2026-01-10 08:30:00'), (5, 'view', TIMESTAMP'2026-01-11 14:45:00'),
  (5, 'view', TIMESTAMP'2026-01-12 09:00:00'), (5, 'purchase', TIMESTAMP'2026-01-12 12:30:00'),
  (5, 'view', TIMESTAMP'2026-01-12 09:00:00')   -- an SDK retry committed this event twice
AS events(user_id, event, ts)""",
    "employees": """CREATE OR REPLACE TEMP VIEW employees AS SELECT * FROM VALUES
  (1,  'Ada',    'Engineering', CAST(NULL AS INT), 185000, DATE'2018-03-01'),
  (2,  'Brett',  'Engineering', 1,    150000, DATE'2019-07-15'),
  (3,  'Chen',   'Engineering', 1,    150000, DATE'2020-01-06'),
  (4,  'Dana',   'Engineering', 2,    155000, DATE'2022-09-12'),   -- out-earns her manager
  (5,  'Eli',    'Sales',       1,    130000, DATE'2019-02-01'),
  (6,  'Farah',  'Sales',       5,    95000,  DATE'2021-05-17'),
  (7,  'Gus',    'Sales',       5,    95000,  DATE'2023-01-09'),
  (8,  'Hana',   'Finance',     1,    140000, DATE'2017-11-20'),
  (9,  'Ivan',   'Finance',     8,    88000,  DATE'2024-04-01'),
  (10, 'Jo',     NULL,          8,    70000,  DATE'2025-06-30')   -- department not assigned yet
AS employees(emp_id, name, dept, manager_id, salary, hire_date)""",
}
for statement in TABLES.values():
    spark.sql(statement)

import contextlib
import io
import logging
import re


def show(query, n=50, **args):
    """Run a query and print up to n rows without truncating values. Keyword arguments bind :name parameters."""
    spark.sql(query, args=args or None).show(n, truncate=False)


def plan(query, mode="formatted"):
    """The plan EXPLAIN prints for a query, as text."""
    return spark.sql(f"EXPLAIN {mode.upper()} {query}").first()[0]


@contextlib.contextmanager
def expected_error():
    """Run a block that is meant to fail and print the error class instead of a stack trace."""
    spark.sparkContext.setLogLevel("OFF")
    for logger in ("DataFrameQueryContextLogger", "SQLQueryContextLogger"):
        logging.getLogger(logger).setLevel(logging.CRITICAL)
    try:
        yield
    except Exception as error:
        lines = [line for line in str(error).splitlines() if line.strip()]
        message = next((line for line in lines if re.search(r"\[[A-Z][A-Z0-9_.]+\]", line)), lines[0] if lines else "")
        print(type(error).__name__ + ":", message.strip().split(". ")[0][:120])
    finally:
        spark.sparkContext.setLogLevel("ERROR")

def fresh(table, select):
    """Drop and re-create a Delta table from a query."""
    spark.sql(f"DROP TABLE IF EXISTS {table}")
    spark.sql(f"CREATE TABLE {table} USING DELTA AS {select}")


def drop(*tables):
    for table in tables:
        spark.sql(f"DROP TABLE IF EXISTS {table}")

## Contents

| section | entries |
| --- | --- |
| [Creating tables](#creating-tables) | [`CREATE TABLE`](#create-table), [Inspecting tables](#inspecting-tables) |
| [Changing tables](#changing-tables) | [`ALTER TABLE` and schema evolution](#alter-table-and-schema-evolution) |
| [Constraints](#constraints) | [`NOT NULL` and `CHECK` constraints](#not-null-and-check-constraints), [Data quality checks in SQL](#data-quality-checks-in-sql) |
| [Views and caching](#views-and-caching) | [Temporary, global and persistent views](#temporary-global-and-persistent-views) |
| [Modeling](#modeling) | [Grain, facts and dimensions](#grain-facts-and-dimensions) |

## Creating tables

### `CREATE TABLE`

```sql
CREATE [OR REPLACE] TABLE [IF NOT EXISTS] name (
    col TYPE [NOT NULL] [DEFAULT expr] [COMMENT '...'], ...
)
USING DELTA | PARQUET | CSV ...
[PARTITIONED BY (cols) | CLUSTER BY (cols)]
[LOCATION 'path']
[COMMENT '...']
[TBLPROPERTIES ('key' = 'value', ...)]
```

| clause | meaning |
| --- | --- |
| `USING` | the table format; Delta (or Iceberg) for ACID updates, constraints and time travel |
| `NOT NULL` | enforced on every write to a Delta table |
| `DEFAULT expr` | value used when an `INSERT` omits the column (Delta needs `'delta.feature.allowColumnDefaults' = 'supported'`) |
| `PARTITIONED BY (col)` | one directory per value: queries filtering on `col` skip whole directories; use only low-cardinality columns |
| `CLUSTER BY (cols)` | Delta liquid clustering: co-locates similar values without fixed directories, can be changed later |
| `LOCATION` | an **external** table at a path you manage; dropping it keeps the files. Without it the table is **managed** and `DROP TABLE` deletes the data |
| `CREATE TABLE new LIKE existing` | copy the schema only |

**Notes:** in open-source Delta, generated and identity columns can only be created through the Python/Scala
`DeltaTable` builder, not `CREATE TABLE` SQL (Databricks supports both in SQL). Partition only large tables, by a column
most queries filter on (typically a date), and keep partitions above roughly 1 GB.

In [ ]:
drop("demo_orders")
spark.sql("""
CREATE TABLE demo_orders (
    order_id    INT           NOT NULL COMMENT 'business key',
    customer_id INT           NOT NULL,
    status      STRING        DEFAULT 'placed',
    order_date  DATE          NOT NULL,
    amount      DECIMAL(12,2)
)
USING DELTA
PARTITIONED BY (order_date)
COMMENT 'Orders, one row per order'
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported')
""")
spark.sql("INSERT INTO demo_orders (order_id, customer_id, order_date, amount) VALUES (1, 1, DATE'2026-01-05', 120.00)")
show("SELECT * FROM demo_orders")
show("DESCRIBE demo_orders")

### Inspecting tables

| command | shows |
| --- | --- |
| `SHOW TABLES [IN db]`, `SHOW VIEWS` | objects in a schema |
| `DESCRIBE t` / `DESCRIBE TABLE EXTENDED t` | columns / plus location, format, owner and properties |
| `DESCRIBE DETAIL t` (Delta) | format, location, partition columns, number of files, size |
| `SHOW COLUMNS IN t`, `SHOW TBLPROPERTIES t` | column names / table properties (including Delta `CHECK` constraints) |
| `DESCRIBE HISTORY t` (Delta) | committed versions |

**Notes:** open-source Spark has no `information_schema` and Delta does not support `SHOW CREATE TABLE`; Databricks
(Unity Catalog), Snowflake, BigQuery and Postgres expose `information_schema` views.

In [ ]:
spark.sql("DESCRIBE DETAIL demo_orders").select("format", "partitionColumns", "numFiles").show(truncate=False)
show("SHOW TABLES")

## Changing tables

### `ALTER TABLE` and schema evolution

| statement | does |
| --- | --- |
| `ALTER TABLE t ADD COLUMNS (c TYPE [COMMENT '...'])` | add nullable columns (existing rows read NULL) |
| `ALTER TABLE t ALTER COLUMN c SET NOT NULL` / `DROP NOT NULL` | change nullability |
| `ALTER TABLE t ALTER COLUMN c COMMENT '...'` | document a column |
| `ALTER TABLE t RENAME COLUMN a TO b`, `DROP COLUMN c` | need Delta column mapping: `'delta.columnMapping.mode' = 'name'` |
| `ALTER TABLE t SET TBLPROPERTIES (...)` | table settings |
| `ALTER TABLE t RENAME TO t2` | rename the table |
| `INSERT ...` with new columns and `spark.databricks.delta.schema.autoMerge.enabled` | automatic schema evolution on write and `MERGE` |

**Notes:** widening types (`INT` to `BIGINT`) requires Delta type widening or a rewrite; narrowing always requires a
rewrite. Additive, nullable changes are the safe kind for downstream consumers.

In [ ]:
spark.sql("ALTER TABLE demo_orders ADD COLUMNS (channel STRING COMMENT 'web or store')")
spark.sql("INSERT INTO demo_orders VALUES (2, 1, 'paid', DATE'2026-01-06', 84.50, 'web')")
show("SELECT order_id, status, amount, channel FROM demo_orders ORDER BY order_id")
spark.sql("ALTER TABLE demo_orders SET TBLPROPERTIES ('delta.columnMapping.mode' = 'name', 'delta.minReaderVersion' = '2', 'delta.minWriterVersion' = '5')")
spark.sql("ALTER TABLE demo_orders RENAME COLUMN channel TO sales_channel")
show("SHOW COLUMNS IN demo_orders")

## Constraints

### `NOT NULL` and `CHECK` constraints

```sql
ALTER TABLE t ADD CONSTRAINT name CHECK (boolean_expression)
ALTER TABLE t DROP CONSTRAINT name
```

| constraint | Delta (open source) | warehouses |
| --- | --- | --- |
| `NOT NULL` | enforced | enforced |
| `CHECK (expr)` | enforced: a write with any violating row fails as a whole; adding a constraint fails if existing rows violate it | Postgres enforces; Snowflake does not support `CHECK` |
| `PRIMARY KEY`, `UNIQUE`, `FOREIGN KEY` | not supported | Postgres enforces; Snowflake, BigQuery and Databricks accept them as **informational** only |

**Notes:** because lakehouse engines do not enforce uniqueness or referential integrity, pipelines check them with
queries (below) and fail the run, or quarantine bad rows, before publishing.

In [ ]:
spark.sql("ALTER TABLE demo_orders ADD CONSTRAINT non_negative_amount CHECK (amount >= 0)")
spark.sql("ALTER TABLE demo_orders ADD CONSTRAINT known_status CHECK (status IN ('placed', 'paid', 'shipped', 'returned'))")
with expected_error():
    spark.sql("INSERT INTO demo_orders VALUES (3, 2, 'paid', DATE'2026-01-07', -5.00, 'web')")
with expected_error():
    spark.sql("INSERT INTO demo_orders VALUES (NULL, 2, 'paid', DATE'2026-01-07', 5.00, 'web')")
spark.sql("SHOW TBLPROPERTIES demo_orders").where("key LIKE 'delta.constraints.%'").show(truncate=False)

### Data quality checks in SQL

Rules that the storage layer cannot enforce become queries that should return **zero rows** (or zero counts):

| rule | check |
| --- | --- |
| uniqueness of a key | `SELECT key, count(*) FROM t GROUP BY key HAVING count(*) > 1` |
| referential integrity | `SELECT ... FROM fact f LEFT ANTI JOIN dim d ON d.key = f.key` |
| not null / domain | `SELECT count_if(col IS NULL), count_if(col NOT IN (...)) FROM t` |
| freshness | `SELECT max(loaded_at) < current_timestamp() - INTERVAL 1 DAY FROM t` |
| volume | compare today's row count with a trailing average |

Run them after each load; dbt tests and Great Expectations generate exactly these queries.

In [ ]:
show("""
SELECT 'orders.order_id unique' AS rule, count(*) AS violations FROM (SELECT order_id FROM orders GROUP BY order_id HAVING count(*) > 1)
UNION ALL
SELECT 'order_items.sku in products', count(*) FROM order_items i LEFT ANTI JOIN products p ON p.sku = i.sku
UNION ALL
SELECT 'customers.region not null', count_if(region IS NULL) FROM customers
""")

## Views and caching

### Temporary, global and persistent views

| object | lifetime | stored |
| --- | --- | --- |
| `CREATE [OR REPLACE] TEMP VIEW v AS ...` | this Spark session | query text only |
| `CREATE GLOBAL TEMP VIEW v AS ...` | the Spark application, shared across sessions as `global_temp.v` | query text only |
| `CREATE [OR REPLACE] VIEW v AS ...` | permanent, in the catalog | query text only; re-runs on every read |
| `CACHE TABLE t` / `CACHE TABLE name AS SELECT ...` | until `UNCACHE` or the session ends | data in executor memory |
| materialised view | Databricks, Snowflake, BigQuery and Postgres | precomputed results, refreshed by the engine |

**Notes:** a view adds no storage and no speed: it is a named query. Use views to give consumers a stable interface
(rename columns, hide internals, apply row filters); use tables or materialised views when the query is expensive and
read often. Temporary views shadow tables with the same name, and a persistent view may only reference persistent
tables and views (`INVALID_TEMP_OBJ_REFERENCE` otherwise), which is why the example below first copies the sample
temp views into Delta tables.

In [ ]:
fresh("customers_t", "SELECT * FROM customers")
fresh("orders_t", "SELECT * FROM orders")
spark.sql("""
CREATE OR REPLACE VIEW v_customer_revenue AS
SELECT c.customer_id, c.name, coalesce(sum(o.amount), 0) AS revenue
FROM customers_t c LEFT JOIN orders_t o ON o.customer_id = c.customer_id AND o.status <> 'returned'
GROUP BY c.customer_id, c.name
""")
show("SELECT * FROM v_customer_revenue ORDER BY revenue DESC LIMIT 3")
spark.sql("CACHE TABLE top_orders AS SELECT * FROM orders WHERE amount > 200")
show("SELECT count(*) AS cached_rows FROM top_orders")
_ = spark.sql("UNCACHE TABLE top_orders")

## Modeling

### Grain, facts and dimensions

| concept | in DDL |
| --- | --- |
| **grain** | the one sentence that defines a row ("one row per order line"); every column must be true at that grain |
| fact table | measures plus foreign keys to dimensions, at the stated grain |
| dimension table | descriptive attributes with one row per key (or per key version for SCD type 2) |
| surrogate key | a warehouse-generated key: a hash of the business key (`xxhash64`, `sha2`) or an identity column |
| date dimension | one row per calendar day with attributes (weekday, month, fiscal period, holiday flags) |

**Notes:** a hash surrogate key is deterministic, needs no lookup, and is the same across environments, but it is
wide and can collide (use `sha2` for very large key spaces). Check the grain after every build: the row count should
equal the count of distinct grain keys.

In [ ]:
fresh("dim_product_m", "SELECT xxhash64(sku) AS product_key, sku, category FROM products")
fresh("fact_order_line", """
SELECT i.order_id, i.sku, xxhash64(i.sku) AS product_key, to_date(o.order_ts) AS order_date, o.customer_id,
       i.qty, i.qty * i.unit_price AS line_amount
FROM order_items i JOIN orders o ON o.order_id = i.order_id
""")
show("SELECT count(*) AS rows, count(DISTINCT order_id, sku) AS grain_keys FROM fact_order_line")
show("""
SELECT coalesce(p.category, 'Unknown') AS category, sum(f.line_amount) AS revenue
FROM fact_order_line f LEFT JOIN dim_product_m p ON p.product_key = f.product_key
GROUP BY coalesce(p.category, 'Unknown') ORDER BY revenue DESC
""")

**Next:** practise these entries in [SQL practice 11: DDL, constraints, views and modeling](sql-practice-11-ddl-constraints-and-views.ipynb).